# Ejemplo de procesos de ciencia de datos

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/i-introduccion-ciencia-datos-programacion/02-ejemplo-procesos-ciencia-datos.ipynb)

## Introducción

Este cuaderno de notas ilustra los procesos de ciencia de datos descritos en la lección de [introducción a la ciencia de datos](01-introduccion-ciencia-datos.md) —importación, estructuración, transformación y visualización—, aplicados a registros de presencia de la lapa verde (*Ara ambiguus*), especie en peligro crítico de extinción. Se utilizan las bibliotecas:

- [pygbif](https://pygbif.readthedocs.io/): acceso a los datos de [GBIF](https://www.gbif.org/) mediante su [interfaz de programación de aplicaciones (API)](https://es.wikipedia.org/wiki/API).
- [pandas](https://pandas.pydata.org/): estructuración y transformación de datos tabulares.
- [plotly](https://plotly.com/python/): gráficos estadísticos interactivos.
- [folium](https://python-visualization.github.io/folium/): mapas interactivos.

La insignia del inicio abre este cuaderno en [Google Colab](https://colab.research.google.com/), donde puede ejecutarse sin instalar nada en la computadora local. Se requiere una cuenta de Google; los requisitos y el acceso se explican en la [guía de Colab del curso](https://gf0657-programacionsig.github.io/2026-ii/colab). Para conservar los cambios, use *Archivo > Guardar una copia en Drive* (*File > Save a copy in Drive*).

### Instalación y carga de bibliotecas

In [1]:
# Instalación de pygbif, necesaria en Google Colab
# (el ambiente conda del curso ya la incluye)
%pip install pygbif --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Carga de bibliotecas
from pygbif import occurrences
import pandas as pd
import plotly.express as px
import folium

# Función auxiliar usada en este cuaderno en lugar de fig.show().
# Despliega el gráfico mediante display(fig), lo que produce una salida
# que el sitio web del curso, Jupyter y Colab muestran de forma interactiva.
from IPython.display import display


def mostrar(fig):
    """Despliega un gráfico de plotly de forma compatible con el sitio del curso."""
    display(fig)

### Parámetros generales

In [3]:
# Nombre científico de la especie
especie = "Ara ambiguus"

## Procesos de ciencia de datos

A continuación se implementan los procesos de importación, estructuración, transformación y visualización descritos en la lección de [introducción a la ciencia de datos](01-introduccion-ciencia-datos.md).

### Importación

Los datos de presencia se obtienen por medio de solicitudes (*requests*) a la API de GBIF mediante la biblioteca pygbif. Como la API entrega los resultados en «páginas» de 300 registros como máximo, se itera hasta recuperarlos todos. Como GBIF recibe registros nuevos continuamente, los resultados varían según la fecha de la consulta (los publicados en el sitio del curso corresponden a agosto de 2026).

In [4]:
limite = 300  # Límite de registros por solicitud
offset = 0  # Desplazamiento para iniciar la solicitud
registros_acumulados = []  # Lista para acumular los resultados

while True:
    # Solicitud
    res = occurrences.search(
        scientificName=especie,
        hasCoordinate=True,
        hasGeospatialIssue=False,
        limit=limite,
        offset=offset
    )

    # Extraer resultados
    registros = res.get("results", [])

    # Si ya no hay resultados, se detiene el ciclo
    if not registros:
        break

    # Agregar registros nuevos al acumulado
    registros_acumulados.extend(registros)

    # Actualizar el offset para la siguiente "página"
    offset += limite

### Estructuración

En este proceso, también llamado *ordenamiento*, los datos se colocan en una estructura rectangular en la que cada fila es una observación y cada columna una variable.

In [5]:
# Convertir los registros acumulados a un DataFrame de pandas
presencia = pd.DataFrame(registros_acumulados)

In [6]:
# Cantidad de registros de presencia recuperados
print(len(presencia))

1525


In [7]:
# Muestra aleatoria, pero reproducible, de los registros recuperados
presencia[['species', 'basisOfRecord', 'countryCode', 'locality', 'decimalLongitude', 'decimalLatitude', 'eventDate', 'year']].sample(10, random_state=42)

,species,basisOfRecord,countryCode,locality,decimalLongitude,decimalLatitude,eventDate,year
782,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-82.821914,9.750305,2022-05-09T15:49,2022.0
76,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-84.398932,10.605405,2026-03-16T10:15,2026.0
1009,Ara ambiguus,HUMAN_OBSERVATION,CR,Tortuguero NP,-83.488380,10.513354,2019-03-16,2019.0
1403,Ara ambiguus,HUMAN_OBSERVATION,CR,Sarapiquí,-84.002435,10.437574,2013-09-03,2013.0
846,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-83.577434,10.498969,2022-10-23T17:04,2022.0
1098,Ara ambiguus,HUMAN_OBSERVATION,CR,San Carlos,-84.180492,10.686970,2018-03-04,2018.0
807,Ara ambiguus,HUMAN_OBSERVATION,CR,La Selva Reserve,-84.005989,10.431014,2022-07-18,2022.0
1495,Ara ambiguus,PRESERVED_SPECIMEN,CO,UPPER RIO BAUDO,-77.333961,4.967590,1940-07-21,1940.0
950,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-82.769080,9.743654,2020-03-03T13:16,2020.0
1188,Ara ambiguus,HUMAN_OBSERVATION,EC,NaN,-80.033528,-2.023539,2017-04-13T12:38,2017.0


### Transformación

En este proceso se generan subconjuntos de las observaciones o de las variables, se crean variables nuevas o se calculan estadísticas. En este caso, se filtran los registros a partir de un año inicial. Nótese que el filtro también excluye los registros sin valor en `year`: otra manifestación de los valores faltantes en los datos reales.

In [8]:
# Año inicial
anio_inicial = 2010

# Filtro por año inicial
presencia = presencia[presencia['year'] >= anio_inicial]

In [9]:
# Cantidad de registros de presencia después del filtro
print(len(presencia))

1442


### Visualización

Como se señaló en la lección, la elección del tipo de gráfico depende del tipo de las variables representadas: el gráfico de barras resume una variable categórica nominal (`countryCode`), el gráfico de líneas muestra la evolución a lo largo de la dimensión temporal (`year`) y el mapa despliega la dimensión espacial, construida a partir de las coordenadas de los registros.

#### Gráfico de barras: registros de presencia por país

In [10]:
# Agrupar por código de país
registros_x_pais = (
    presencia
    .groupby('countryCode', dropna=True)
    .size()
    .reset_index(name='cantidad')
)

# Ordenar descendentemente por frecuencia
registros_x_pais = registros_x_pais.sort_values('cantidad', ascending=False)

# Crear gráfico de barras
fig = px.bar(
    registros_x_pais,
    x='countryCode',
    y='cantidad',
    title='Cantidad de registros de presencia por país',
    labels={'countryCode': 'País', 'cantidad': 'Cantidad de registros de presencia'},
    text='cantidad'
)

# Personalizar tooltip (hover) y estilo de las barras
fig.update_traces(
    hovertemplate='País: %{x}<br>Cantidad de registros de presencia: %{y}',
    marker_color='rgb(55, 83, 109)'
)

# Opciones de diseño
fig.update_layout(
    template='plotly_white',
    xaxis={'type': 'category'},
    xaxis_title='País',
    yaxis_title='Cantidad de registros de presencia',
    showlegend=False,
    annotations=[
        dict(
            text='Fuente: GBIF',
            x=1, y=1,
            xref='paper', yref='paper',
            xanchor='right', yanchor='bottom',
            showarrow=False
        )
    ]
)

# Mostrar la figura
mostrar(fig)

#### Gráfico de líneas: registros de presencia por año

In [11]:
# Agrupar por año
registros_x_anio = (
    presencia
    .groupby('year', dropna=True)
    .size()
    .reset_index(name='cantidad')
)

# Crear gráfico de líneas
fig = px.line(
    registros_x_anio,
    x='year',
    y='cantidad',
    labels={'year': 'Año', 'cantidad': 'Cantidad de registros de presencia'},
    title='Cantidad de registros de presencia por año'
)

# Personalizar tooltip (hover) y estilo de la línea
fig.update_traces(
    mode='lines+markers',
    hovertemplate=(
        "Año: %{x}<br>"
        "Cantidad de registros de presencia: %{y}"
    )
)

# Opciones de diseño
fig.update_layout(
    template='plotly_white',
    annotations=[
        dict(
            text='Fuente: GBIF',
            x=1, y=0,
            xref='paper', yref='paper',
            xanchor='right', yanchor='bottom',
            showarrow=False
        )
    ],
    xaxis_title='Año',
    yaxis_title='Cantidad de registros de presencia'
)

# Mostrar la figura
mostrar(fig)

#### Mapa: ubicación de los registros de presencia

In [12]:
# Crear el mapa base, centrado en el área de distribución de la especie
m = folium.Map(
    location=[8.5, -81],
    zoom_start=5,
    tiles='OpenStreetMap',
    name='Mapa general'
)

# Agregar capa de teselas de Esri WorldImagery (imágenes satelitales)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Imágenes satelitales',
    overlay=False,
    control=True
).add_to(m)

# Agregar capa de teselas de CartoDB positron (mapa en blanco)
folium.TileLayer(
    'CartoDB positron',
    name='Mapa en blanco',
    overlay=False,
    control=True
).add_to(m)

# Crear un FeatureGroup para los registros de presencia
fg_presencia = folium.FeatureGroup(name='Registros de presencia')

# Iterar sobre el DataFrame y agregar cada punto
for i, row in presencia.iterrows():
    # Extraer coordenadas
    lat = row['decimalLatitude']
    lon = row['decimalLongitude']

    # Verificar que lat y lon sean válidos
    if pd.notnull(lat) and pd.notnull(lon):
        # Construir popup en HTML
        popup_html = (
            f"<strong>País: </strong>{row.get('country','')}<br/>"
            f"<strong>Localidad: </strong>{row.get('locality','')}<br/>"
            f"<strong>Fecha: </strong>{row.get('eventDate','')}<br/>"
            f"<strong>Fuente: </strong>{row.get('institutionCode','')}<br/>"
            f"<a href='https://www.gbif.org/occurrence/{row.get('key','')}' target='_blank'>Más información</a>"
        )

        # Agregar marcador circular
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            fill=True,
            fill_color='red',
            fill_opacity=1,
            stroke=False,
            popup=popup_html
        ).add_to(fg_presencia)

# Agregar el FeatureGroup al mapa
fg_presencia.add_to(m)

# Agregar el control de capas
folium.LayerControl().add_to(m)

# Mostrar el mapa
m

## Ejercicios

1. Abra este cuaderno en Google Colab mediante la insignia del inicio y guarde su propia copia (*Archivo > Guardar una copia en Drive*). Verifique que puede ejecutarlo completo (*Entorno de ejecución > Ejecutar todas*); los siguientes ejercicios se realizan sobre esa copia.
2. En la sección de parámetros generales, cambie el valor de `especie` por el nombre científico de otra especie de su interés y vuelva a ejecutar todo el cuaderno. Antes de elegirla, consulte en [GBIF](https://www.gbif.org/) cuántos registros de presencia tiene: una especie con decenas de miles de registros tardará más en descargarse y producirá un mapa pesado.
3. Modifique el valor de `anio_inicial` en la sección de transformación y observe el efecto en la cantidad de registros y en las visualizaciones.
4. Agregue la variable `elevation` a la muestra de registros de la sección de estructuración. ¿Cuántos de los registros de la muestra tienen valores faltantes en esa variable?